# 05 — Interferometers: amplitudes, not probabilities

## What you will learn

Richard Feynman opened his lectures on quantum mechanics by describing the double-slit
experiment and then saying that it

> "has in it the heart of quantum mechanics. In reality, it contains the *only* mystery."

Everything else, he claimed, is either bookkeeping or a consequence. This notebook takes
him at his word and runs the experiment — along with four of its descendants, each of
which was a landmark paper when it was first thought of.

Here is the surprise, and it is the reason this notebook needs no new library machinery:
**you have been building interferometers since notebook 01 without being told.**

- A **50/50 beam splitter** — the half-silvered mirror that sends a photon both ways at
  once — is a **Hadamard**.
- A **phase shifter** in one arm — a slab of glass that slows the light down — is
  `Rz(path, theta=φ)`.
- So `H`, then `Rz`, then `H` is a **Mach–Zehnder interferometer**, one of the workhorse
  instruments of experimental optics. The H-sandwich you have run a dozen times *was*
  one.
- A **which-path detector** is a `CNOT` onto a spare qubit — which is precisely the
  "dirty ancilla" that wrecked the interference at the end of notebook 04. That was a
  which-path experiment, and nobody told you its name.

Nothing below is a new capability. It is the same gates wearing optical labels, which is
worth pausing on: the reason a qubit simulator can run optics experiments is that neither
one is really about photons or about qubits. Both are about **amplitudes adding**.

By the end you will know:

- why a Mach–Zehnder interferometer sends every photon out of the *same* port even though
  both of its beam splitters are perfectly fair — and why no story told in probabilities
  can account for that;
- **complementarity** stated as an equation rather than a slogan: $V^2 + D^2 = 1$, where
  $V$ is how sharp the fringes are and $D$ is how much you could know about the path.
  Fringe contrast and which-path knowledge are not two effects; they are one budget;
- the **Elitzur–Vaidman bomb tester**: how to certify that a bomb is live using a photon
  that never went near it — a thing classical physics says is flatly impossible;
- what happens with **more than two paths**, which turns out to be the engine every
  quantum algorithm in the second half of this project runs on;
- **Stern–Gerlach filters** in series, the experiment Feynman actually opens Volume III
  with, where a third filter along an axis you already filtered for throws away half your
  atoms;
- **delayed choice**: deciding whether to erase the which-path record *after* the photon
  is already past the beam splitter — and a careful account of what that does and does
  not prove.

A word on what an interferometer physically *is*, since the qubit version hides the
hardware.

A photon arrives at a half-silvered mirror. Half the light goes through, half bounces off
— but a single photon is not half a photon, and it does not choose. It leaves the mirror
in a superposition of "went straight on" (call that arm 0) and "was reflected" (arm 1).
Two ordinary mirrors bring the two arms back together at a second half-silvered mirror,
and two detectors sit at the two outputs. Somewhere along one arm we can put a piece of
glass, which delays that part of the wave and gives it an extra phase $\varphi$ relative
to the other.

```text
                       mirror ┌──────────────── arm 1 ──────────────┐
                              │                                     ▼
   photon ──▶ ╱ beam splitter 1 ──── arm 0 ──▶ [ glass: phase φ ] ──▶ ╱ beam splitter 2 ──▶ port 0
              ╱                                                      ╱  │
                                                                        ▼
                                                                      port 1
```

In qsim all of that geometry collapses into **one qubit**, whose two basis states are the
two arms. $|0\rangle$ means "in arm 0", $|1\rangle$ means "in arm 1", and a superposition
means what a superposition always means. The mirrors and the table and the lasers are
scenery; the physics is entirely in the two amplitudes.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, viz
from qsim.algorithms.interferometry import (
    bomb_probabilities,
    bomb_test,
    distinguishability,
    filter_chain,
    fringes,
    mach_zehnder,
    n_path_fringes,
    visibility,
)
from qsim.gates import H, Ry, Rz

np.set_printoptions(precision=3, suppress=True)

## 1. The beam splitter is a Hadamard

Here is the whole translation table, which is the only thing you need in order to read
optics papers as qsim programs:

| apparatus | qsim |
|---|---|
| a 50/50 beam splitter | `H` |
| a phase shifter in one arm | `Rz(path, theta=φ)` |
| which path the photon took | the state of one qubit, $|0\rangle$ or $|1\rangle$ |
| a which-path detector | `CNOT(path, detector)` |
| a *partial* which-path detector | `with qc.control(path): Ry(detector, theta=θ)` |
| the two output ports | the two outcomes of measuring the path qubit |

Why is a beam splitter a Hadamard? Because $H$ is exactly the operation "take whichever
arm you are in and put half the amplitude in each arm, with a sign":

$$H|0\rangle = \tfrac{1}{\sqrt2}\big(|0\rangle + |1\rangle\big), \qquad
  H|1\rangle = \tfrac{1}{\sqrt2}\big(|0\rangle - |1\rangle\big).$$

Both outputs have magnitude $1/\sqrt2$, so the splitter is scrupulously fair either way
you enter it — a photon coming in on arm 0 and a photon coming in on arm 1 both leave
50/50. The *only* difference between the two rows is a minus sign, which no probability
can see. Hold on to that minus sign; the entire notebook is about it.

Let us build the instrument by hand, printing the state after every optical component.

In [ ]:
qc = Circuit(name="Mach-Zehnder")
path = qc.alloc("path")

print("photon enters:        ", qc.inspect.ket())

H(path)                    # beam splitter 1: one photon, both arms
print("after beam splitter 1:", qc.inspect.ket())

Rz(path, theta=np.pi / 2)  # a slab of glass in one arm: phase φ = π/2
print("after the phase shift:", qc.inspect.ket())

H(path)                    # beam splitter 2: the two arms meet again
print("after beam splitter 2:", qc.inspect.ket())

print()
print("P(photon leaves by port 0) =", qc.inspect.probabilities()[0])
print("the packaged version       =", mach_zehnder(np.pi / 2))

Three things in that printout are worth reading slowly.

**After beam splitter 1** the photon is in $(|0\rangle + |1\rangle)/\sqrt2$: both arms,
amplitude $0.707$ each. Not "it is in one arm and I don't know which" — we will disprove
that reading twice before this notebook is over.

**After the phase shift** the two amplitudes are $0.5 - 0.5i$ and $0.5 + 0.5i$. Both
still have magnitude $0.707$, so *nothing measurable has changed*: the probability of
finding the photon in either arm is still one half. All that happened is that the two
amplitudes rotated in the complex plane, in opposite directions. `Rz(theta=φ)` multiplies
$|0\rangle$ by $e^{-i\varphi/2}$ and $|1\rangle$ by $e^{+i\varphi/2}$; splitting the
angle symmetrically like that is a convention, and only the **difference** $\varphi$
between the two arms has any physical meaning.

**After beam splitter 2** the amplitudes are $0.707$ and $-0.707i$: fifty-fifty again, at
this particular $\varphi$. The interesting values of $\varphi$ are elsewhere.

Since the phase is invisible to probabilities but decisive for the outcome, this is the
one place in the notebook where the phase-colored bar chart earns its keep. The two bars
below are the same height — the state is 50/50 in the arms — and different colors.
**The colors are what the second beam splitter acts on.**

In [ ]:
qc = Circuit(name="inside the interferometer")
path = qc.alloc("path")
H(path)
Rz(path, theta=np.pi / 2)  # stop the photon mid-flight, between the two splitters

fig = viz.amplitudes(qc)   # bar height = |amplitude|, bar color = its phase

Now turn the glass. At $\varphi = 0$ the two arms are identical; at $\varphi = \pi$ they
are exactly out of step.

In [ ]:
for name, phi in [("0", 0.0), ("π/2", np.pi / 2), ("π", np.pi)]:
    p0 = mach_zehnder(phi)
    print(f"φ = {name:>4}:   P(port 0) = {p0:.6f}    P(port 1) = {1 - p0:.6f}")

Read those numbers again, because they should be impossible.

Every photon enters the same way. Both beam splitters are perfectly fair — each one, on
its own, sends a photon either way with probability one half. And yet at $\varphi = 0$
**every single photon comes out of port 0** and port 1 is *completely dark*. Turn the
glass to $\varphi = \pi$ and the situation reverses exactly: port 0 goes dark instead.

Here is the classical bookkeeping, done carefully, for port 1 at $\varphi = 0$. There are
two routes a photon could take to port 1:

- take arm 0 (probability $\tfrac12$), then be sent to port 1 by splitter 2
  (probability $\tfrac12$) — total $\tfrac14$;
- take arm 1 (probability $\tfrac12$), then be sent to port 1 by splitter 2
  (probability $\tfrac12$) — total $\tfrac14$.

Probabilities of alternatives add: $\tfrac14 + \tfrac14 = \tfrac12$. Port 1 should fire
half the time, for every setting of the glass. It fires **never**.

Now do it with amplitudes, which are what actually add:

- via arm 0: $\tfrac{1}{\sqrt2} \times \big(+\tfrac{1}{\sqrt2}\big) = +\tfrac12$;
- via arm 1: $\tfrac{1}{\sqrt2} \times \big(-\tfrac{1}{\sqrt2}\big) = -\tfrac12$
  — the minus sign from $H|1\rangle = (|0\rangle - |1\rangle)/\sqrt2$.

Total amplitude at port 1: $+\tfrac12 - \tfrac12 = 0$. Square it: probability zero. The
two routes are still there, both perfectly possible, and they **cancel**.

This is the whole of quantum mechanics in one arithmetic step. Probabilities are
non-negative, so a sum of them can only ever grow: adding a second way for something to
happen can never make it happen less often. Amplitudes are complex, so a second way can
subtract. **You cannot get this from any theory in which the photon took one arm and you
merely failed to notice which** — that theory is the one we just did, and it predicts
$\tfrac12$.

Sweeping the glass traces out the **interference fringes**.

In [ ]:
phis = np.linspace(-np.pi, np.pi, 201)
pattern = fringes(phis)  # mach_zehnder() at each phase

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(phis, pattern, linewidth=2, color="tab:blue", label="qsim: P(port 0)")
ax.plot(phis, np.cos(phis / 2) ** 2, "--", linewidth=1.4, color="black",
        label="cos²(φ/2)")
ax.axhline(0.5, color="crimson", linestyle=":", linewidth=1.6,
           label="what adding probabilities predicts")
ax.set_xlabel("φ — extra phase in one arm")
ax.set_ylabel("P(the photon leaves by port 0)")
ax.set_title("Mach–Zehnder fringes: both splitters are 50/50")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["−π", "−π/2", "0", "π/2", "π"])
ax.set_ylim(-0.03, 1.08)
ax.legend(loc="lower center", fontsize=9)

print("largest disagreement with cos²(φ/2):",
      float(np.max(np.abs(pattern - np.cos(phis / 2) ** 2))))

$P(\text{port }0) = \cos^2(\varphi/2)$, to fifteen decimal places, and it swings all the
way from $1$ to $0$. The dotted red line is the classical prediction: flat, featureless,
wrong.

The fringes are what an interferometer is *for*. Because the output depends on the phase
difference between the arms, and because phase is what changes when a path length changes
by a fraction of a wavelength, an interferometer converts a distance you cannot measure
into a brightness you can. That is how LIGO detected gravitational waves: a Michelson
interferometer with 4-km arms, reading out a length change smaller than a proton.
Everything in this notebook is a laboratory instrument first and a paradox second.

## 2. Which path? — complementarity as a conservation law

The obvious next question is the one everybody asks: *fine, but which arm did it actually
go down?* So look. Put a detector in one arm.

The detector is a spare qubit that gets flipped when the photon comes past. That is a
`CNOT` from the path qubit onto the detector qubit — and you have already watched exactly
this circuit, at the end of notebook 04, where the same `CNOT` onto a scratch qubit was a
bug we were trying to avoid. Same gates, same state, different story:

| notebook 04 | this notebook |
|---|---|
| an ancilla that was not uncomputed | a which-path detector |
| `DirtyAncillaError` | a successful measurement |
| interference destroyed — a bug | interference destroyed — the data |

The library makes the detector adjustable, which is the part worth having. Instead of a
`CNOT` — which flips the detector all the way — use a **controlled rotation** by an angle
$\theta$: `with qc.control(path): Ry(detector, theta=θ)`. At $\theta = 0$ the detector
does not move and learns nothing. At $\theta = \pi$ it flips completely and learns
everything, which is just the `CNOT` again. In between it ends up in two states that are
neither the same nor opposite — it learns *something*. Which-path information is a dial,
not a switch, and the whole of this section lives on that dial.

In [ ]:
labels = [("no detector (θ = 0)", 0.0),
          ("θ = π/4", np.pi / 4),
          ("θ = π/2", np.pi / 2),
          ("θ = 3π/4", 3 * np.pi / 4),
          ("perfect detector (θ = π)", np.pi)]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for name, theta in labels:
    ax.plot(phis, fringes(phis, detector_strength=theta), linewidth=2, label=name)
ax.set_xlabel("φ — extra phase in one arm")
ax.set_ylabel("P(the photon leaves by port 0)")
ax.set_title("the more the detector knows, the flatter the fringes")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["−π", "−π/2", "0", "π/2", "π"])
ax.legend(fontsize=9)

for name, theta in labels:
    swept = fringes(phis, detector_strength=theta)
    print(f"{name:<26} max = {swept.max():.4f}   min = {swept.min():.4f}")

The fringes fade, smoothly, and at $\theta = \pi$ they are gone: a flat $0.5$ at every
phase, which is the classical prediction from section 1 finally coming true. Turning the
glass no longer does anything at all.

And notice what we did **not** do. We never read the detector. There is no `qc.measure`
anywhere in this section, no `if` branching on the result, no human looking at a dial.
The detector qubit simply sits there, correlated with the path. That is enough. As
notebook 04 put it: the two routes to port 1 now end in *different total states* —
$|{\rm arm}\,0\rangle|{\rm detector\ not\ flipped}\rangle$ versus
$|{\rm arm}\,1\rangle|{\rm detector\ flipped}\rangle$ — and different states cannot
cancel. The minus sign is still there. It just has nothing to cancel against.

### Two numbers, one budget

Define the two quantities the fading is a trade between.

**Visibility** $V$ measures how sharp the fringes are:

$$V = \frac{P_{\max} - P_{\min}}{P_{\max} + P_{\min}}$$

which is $1$ for perfect fringes (dark to bright) and $0$ for a flat line. Since our
fringes are symmetric about $\tfrac12$, the denominator is $1$ and $V$ is just the
peak-to-trough swing you can read straight off the plot above.

**Distinguishability** $D$ measures how well the detector's final state reveals the arm.
Two detector states that are identical give $D = 0$; two orthogonal ones — perfectly
tellable apart, in principle, by *someone*, using the best measurement allowed by physics
— give $D = 1$.

The library computes them as $V = |\cos(\theta/2)|$ and $D = |\sin(\theta/2)|$, which are
the overlap and the anti-overlap of the two detector states. Plot them together.

In [ ]:
thetas = np.linspace(0.0, np.pi, 61)
V = np.array([visibility(t) for t in thetas])
D = np.array([distinguishability(t) for t in thetas])

# V read off the actual fringe pattern, rather than from the formula: peak minus trough.
V_measured = np.array([np.ptp(fringes(phis, detector_strength=t)) for t in thetas])

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(thetas, V, linewidth=3, color="tab:blue", label="V — fringe visibility")
# Every third measured point, so the formula's line stays visible underneath them.
ax.plot(thetas[::3], V_measured[::3], "o", markersize=4, color="black",
        label="V measured off the fringes")
ax.plot(thetas, D, linewidth=3, color="crimson", label="D — which-path knowledge")
ax.plot(thetas, V**2 + D**2, linewidth=2.5, color="seagreen", linestyle="--",
        label="V² + D²")
ax.set_xlabel("θ — how hard the detector looks")
ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
ax.set_xticklabels(["0", "π/4", "π/2", "3π/4", "π"])
ax.set_ylim(-0.05, 1.5)
ax.set_title("complementarity: V² + D² = 1")
ax.legend(fontsize=9, loc="upper center", ncol=2)

print("largest deviation of V²+D² from 1:", float(np.max(np.abs(V**2 + D**2 - 1))))
print("largest gap between formula V and measured V:",
      float(np.max(np.abs(V - V_measured))))

The green line is flat at $1$ to machine precision:

$$V^2 + D^2 = 1.$$

That is **complementarity**, and stating it this way is a considerable improvement on the
usual slogan about wave-particle duality. It is not a rule that you must choose between
two pictures, nor a warning that measurement "disturbs" things. It is a **conservation
law**. There is exactly one unit of resource, and it can be spent on fringe contrast, on
which-path knowledge, or split between them in any proportion — but the total is fixed at
one, always, and nothing you can build gets more.

Two consequences worth spelling out:

- **Partial knowledge costs partial contrast.** At $\theta = \pi/2$, $V = D = 0.707$: the
  detector is right about the path more often than chance, and the fringes are still
  visible, just faded. Nothing is all-or-nothing here.
- **The knowledge does not have to be collected.** $D$ is defined by how distinguishable
  the two detector states *are*, not by whether anyone distinguishes them. The fringes
  are already gone at the moment the coupling happens. A record that exists but is never
  read costs exactly as much as one that is read — which is why notebook 04's ancilla
  check is a physical requirement and not a tidiness rule, and why notebook 06's
  environment, which nobody could read even in principle, still ruins everything.

(In the general case complementarity is an *inequality*, $V^2 + D^2 \le 1$, with equality
only for pure states. Our interferometer is pure — nothing has been discarded — so we sit
exactly on the boundary. Any additional noise pushes you underneath it, losing contrast
without buying knowledge: the worst of both worlds, and also the ordinary situation in a
real laboratory.)

## 3. The bomb: learning about a thing your photon never touched

In 1993 Avshalom Elitzur and Lev Vaidman asked a question that sounds like a riddle and
turns out to be a laboratory procedure. Here is their setup.

You have a crate of bombs. Each bomb has a trigger so sensitive that **absorbing a single
photon sets it off**. Some of the bombs in the crate are duds: the trigger is broken, the
photon sails straight through, nothing happens. You would like to find a bomb that you
*know* is live.

Classically this is hopeless, and it is worth being precise about why. The only physical
difference between a live bomb and a dud is what the trigger does to a photon that hits
it. To learn anything about the trigger you must send a photon at it. If the bomb is
live, that photon is absorbed and the bomb explodes. **Any test that could return "live"
destroys the object it was testing.** There is no classical procedure — none, not a clever
one, not an expensive one — that ends with an intact bomb you know to be live.

Now put the bomb in **arm 1** of the interferometer.

A live bomb absorbs any photon that comes down its arm. That means the bomb *registers
which path the photon took*: bomb intact tells you "arm 0", bomb exploded tells you "arm
1". A live bomb is therefore a which-path detector — a perfect one, $\theta = \pi$ — and
by section 2 it destroys the interference. A dud registers nothing, is not a detector,
and the fringes survive.

Set the glass to $\varphi = 0$, where port 1 is completely dark. Then:

- **Dud:** interference intact, port 1 dark. Every photon leaves by port 0.
- **Live:** interference destroyed. Half the time the photon goes down arm 1 and the bomb
  explodes. The other half it goes down arm 0, survives, and leaves by port 0 or port 1
  with equal probability — because with the interference gone, nothing keeps port 1 dark
  any more.

So **a click at port 1 can only happen if the bomb is live** — and in exactly those runs
the photon went down the *other* arm and the bomb absorbed nothing at all.

In [ ]:
live = bomb_probabilities()
dud = bomb_probabilities(live=False)

print(f"{'outcome':<16}{'live bomb':>12}{'dud':>12}")
for key in ("exploded", "found", "inconclusive"):
    print(f"{key:<16}{live[key]:>12.4f}{dud[key]:>12.4f}")

Half the time the bomb goes off — that is the price, and it is a real one. A quarter of
the time you get a click at port 1: **the bomb is live and it is still sitting there.**
The remaining quarter is a click at port 0, which tells you nothing (a dud does that too),
so you set that bomb aside and try again with the next photon.

And look at the dud column: `found` is exactly zero. Port 1 is *never* reached when the
interference is intact, so a port-1 click is not evidence-in-the-Bayesian-sense; it is a
proof. One click, one certified live bomb, zero explosions.

Now sample it. Each `bomb_test(seed=...)` is one photon through one interferometer,
measured — the bomb qubit first ("did it go off?"), then the port.

In [ ]:
one = bomb_test(seed=0)
print("a single run:", one)
print()

runs_live = Counter(bomb_test(seed=s).outcome for s in range(2000))
runs_dud = Counter(bomb_test(live=False, seed=s).outcome for s in range(2000))

print(f"{'outcome':<16}{'live (of 2000)':>16}{'dud (of 2000)':>16}{'theory, live':>14}")
for key in ("exploded", "found", "inconclusive"):
    print(f"{key:<16}{runs_live[key]:>16}{runs_dud[key]:>16}{live[key]:>14.2f}")

Roughly 1000 explosions, 500 certified live bombs, 500 wasted photons — the $1/2$, $1/4$,
$1/4$ split, sampled. The dud column is 2000 inconclusive runs and not a single port-1
click, which is the claim the whole scheme rests on.

### The honest caveat

It is very easy to over-tell this story, so here is the careful version.

In the runs that end at port 1, **no photon was absorbed by the bomb**. That much is
exactly true, and it is why the bomb is still there afterwards. What is *not* true is the
tempting summary "the photon went down the empty arm, so nothing about the bomb could
have mattered". Something about the bomb mattered enormously.

What the bomb changed was not the photon's path. It was **what was possible**. The
interference that kept port 1 dark was a cancellation between two amplitudes — one for
each arm — and cancellation requires that the two routes end in *indistinguishable*
states. A live bomb sitting in arm 1 makes them distinguishable: one route ends with the
bomb intact, the other with the bomb blown apart. That is enough to break the
cancellation, and it is enough whether or not the photon "goes there", because the
amplitude for the arm-1 route is part of the calculation regardless of what any individual
photon "did".

This is the same lesson as section 2, delivered with more drama. The thing that destroys
interference is not an interaction, and it is not an observation. It is the existence of a
difference — anywhere in the world — between the two ways the story could have gone.

Elitzur and Vaidman called this **interaction-free measurement**, and it is not a thought
experiment: Kwiat, Weinfurter and Zeilinger ran it in 1995, and a refinement of the scheme
(nesting interferometers, so that the explosion probability can be pushed arbitrarily
close to zero) has since been used to image objects with light that never hit them.

## 4. More than two paths

Two slits give a broad cosine. What does a diffraction *grating* — many slits — give?

`n_path_fringes(k, phases)` builds an interferometer with $2^k$ paths instead of two: $k$
path qubits, all Hadamarded, so that every one of the $2^k$ basis states is a route the
photon can take. Then path $j$ is given a phase of $j\varphi$, which costs one `Rz` per
qubit — qubit $i$ carries place value $2^{k-1-i}$, so a phase proportional to that place
value on each qubit makes path $j$ pick up exactly $j\varphi$. Finally another Hadamard
on each qubit brings all $2^k$ routes back together, and we ask how much amplitude
returns to "every path qubit reads 0".

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for k in (1, 2, 3):
    ax.plot(phis, n_path_fringes(k, phis), linewidth=2, label=f"{2**k} paths")
ax.set_xlabel("φ — phase step between neighbouring paths")
ax.set_ylabel("P(the photon arrives at the bright spot)")
ax.set_title("more paths: the peak stays at 1 and gets narrower")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["−π", "−π/2", "0", "π/2", "π"])
ax.legend(fontsize=9)

step = float(phis[1] - phis[0])
for k in (1, 2, 3):
    curve = n_path_fringes(k, phis)
    width = float((curve > 0.5).sum()) * step
    print(f"{2**k:>2} paths:  peak = {curve.max():.6f}   width at half height = "
          f"{width:.3f} rad")

The peak never gets any taller — it is already at $1$, meaning *every* photon lands there
when the phases line up — but it gets **narrower** each time the number of paths doubles,
and small ripples appear on either side.

The mechanism is worth saying in one sentence, because the rest of this project runs on
it. At $\varphi = 0$ all $2^k$ amplitudes point the same way in the complex plane and add
up constructively. Move $\varphi$ slightly off zero and the amplitudes fan out around a
circle; the more of them there are, the more completely they cancel, and the smaller a
nudge it takes to make them do it. Many paths means sharp cancellation everywhere except
where the phases exactly agree.

**That is a computation.** You have $2^k$ routes and one output; if you can arrange for
the phase of each route to encode something about a candidate answer, then the routes that
correspond to wrong answers cancel, and the one that corresponds to the right answer
survives. This is precisely what the **quantum Fourier transform** does — notebook 07 —
and the sharpening you just watched is why period-finding gets *more* precise as you add
qubits rather than merely faster. Shor's algorithm is a diffraction grating aimed at
arithmetic.

## 5. Stern–Gerlach filters: the experiment Feynman opens with

Volume III of the *Feynman Lectures* does not begin with photons. It begins with a beam
of silver atoms and a magnet.

A **Stern–Gerlach apparatus** is an inhomogeneous magnetic field that deflects atoms
according to their spin along the field axis. Send in a beam and it splits into two — not
a smear, which is what a classical spinning magnet would give, but two discrete spots.
Spin along an axis has exactly two values. Block one of the two outputs with a piece of
metal and you have a **filter**: whatever emerges is definitely spin-up along that axis.

In qsim a filter is exactly what that description says: *rotate the apparatus to the axis
you want, measure, and throw away the atoms that came out of the blocked port*.
`filter_chain` does this atom by atom — one `Circuit` per atom, walked through the whole
chain, discarded the moment a filter blocks it. Measuring along a tilted axis is the trick
from notebook 03 and from `chsh.py`: rotate the state by $-\alpha$ and then measure $z$,
which is the same experiment as measuring along the axis $\alpha$.

Chain some filters and count survivors.

In [ ]:
shots = 2000
for chain in ("z", "zz", "zx", "zxz"):
    counts = filter_chain(chain, shots, seed=1)
    stages = "  →  ".join(f"{c}" for c in counts)
    print(f'filter_chain("{chain}"):   {stages}')
print()
print("(the first number is the atoms that left the oven; each later number is how many")
print(" survived the corresponding filter)")

Take those four rows one at a time.

**`"z"`** — one filter along $z$. Everything survives, because the atoms in this
simulation leave the oven already spin-up along $z$ (a real oven emits both, and the first
filter is what prepares the beam; qsim starts every qubit at $|0\rangle$, which is
spin-up along $z$, so the preparation is free).

**`"zz"`** — filter along $z$, then along $z$ again. Everything survives again. This is
the reassuring, classical-feeling result: the second filter asks a question the first one
has already answered, and gets the same answer. The atoms *have* a definite $z$-spin, and
asking twice does not change it.

**`"zx"`** — filter along $z$, then along $x$. About half survive. Also unsurprising:
"definitely up along $z$" says nothing whatever about $x$, so the $x$-filter is a coin
flip. In notebook 03's language, $|0\rangle$ written in the $x$ basis is
$(|{+}\rangle + |{-}\rangle)/\sqrt2$.

**`"zxz"`** — and now the one that broke everybody's intuition in 1922. The third filter
is along $z$, *the very axis we filtered for at the start*. These atoms were certified
spin-up along $z$ by filter 1. Filter 2 did not touch $z$; it asked about $x$, and we kept
only the atoms that passed. And yet **half of them now fail the $z$ filter**.

The atoms that come out at the end were, at some point, certified up along $z$, then
certified up along $x$ — and by the end, half of them are down along $z$. The $x$
measurement did not merely *select* a subset of a population that already had definite
values of everything. **It destroyed the $z$ information.** There is no assignment of a
$z$-value and an $x$-value to each atom, fixed in advance and merely revealed by the
filters, that reproduces these three numbers.

This is the same fact as sections 1–3, arriving from a completely different direction —
here it looks like the failure of a Venn diagram rather than the cancellation of two
amplitudes. It is the reason notebook 03 could measure the *same* qubit along a rotated
axis and get a different answer, and it is one Hadamard away from the interferometer: $H$
is exactly the change of basis between the $z$ axis and the $x$ axis, which is why the
same gate is both "a beam splitter" and "turn the magnet 90°".

## 6. Delayed choice: erasing the record after the fact

John Wheeler asked the following in 1978. The photon meets the first beam splitter and
"decides" — so the story goes — whether to behave as a wave that takes both arms or as a
particle that takes one. Suppose we wait until *after* it has passed that splitter, and
only then decide what kind of experiment to run. Does the photon somehow have to go back
and choose again?

We can run precisely that. Build the interferometer by hand, couple a detector to the
path so the which-path record is written, and then — with the photon already past the
splitter, the phase already applied, the record already on the detector qubit — decide in
ordinary Python whether to erase it, using `with qc.adjoint():` on the coupling. Notebook
04 built that scope: it reverses the operations and inverts each one, so it takes the
detector qubit back to $|0\rangle$ regardless of which arm the photon is in, wiping the
correlation.

In [ ]:
def delayed_choice(phase: float, *, erase: bool) -> float:
    """One photon through a Mach-Zehnder, with the erasure decided after the fact."""
    qc = Circuit(name="delayed-choice")
    path = qc.alloc("path")
    detector = qc.alloc("detector")

    H(path)                    # beam splitter 1 — the photon is now in both arms
    Rz(path, theta=phase)      # the glass
    with qc.control(path):     # the detector learns the path, perfectly (θ = π)
        Ry(detector, theta=np.pi)

    # ---- the photon is past the splitter and the record is already written ----
    if erase:
        with qc.adjoint():     # run the coupling backwards: unwrite the record
            with qc.control(path):
                Ry(detector, theta=np.pi)

    H(path)                    # beam splitter 2
    # The path qubit's reduced density matrix ignores the detector entirely, which is
    # what a detector sitting at port 0 actually sees: it has no access to the other qubit.
    return float(qc.inspect.reduced_density_matrix([path])[0, 0].real)


kept = np.array([delayed_choice(float(p), erase=False) for p in phis])
erased = np.array([delayed_choice(float(p), erase=True) for p in phis])

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(phis, kept, linewidth=2, color="crimson", label="record kept — no fringes")
ax.plot(phis, erased, linewidth=2, color="tab:blue", label="record erased — fringes back")
ax.set_xlabel("φ — extra phase in one arm")
ax.set_ylabel("P(the photon leaves by port 0)")
ax.set_title("the choice is made after the photon is already inside")
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["−π", "−π/2", "0", "π/2", "π"])
ax.set_ylim(-0.03, 1.08)
ax.legend(fontsize=9, loc="lower center")

print("record kept  : max", round(kept.max(), 6), " min", round(kept.min(), 6))
print("record erased: max", round(erased.max(), 6), " min", round(erased.min(), 6))

Full fringes, restored by a decision made in Python after the coupling had already run.

### What this shows, and what it does not

It does **not** show retrocausality. Nothing travelled backwards in time, and the photon
did not revise a choice it had made earlier. Two things make that clear, and both are
visible in the code:

1. **The photon never made a choice to revise.** There is no moment in the circuit at
   which the path qubit "became" one arm or the other. It was in a superposition of both
   the entire time; that is all `H` ever did to it. The wave-or-particle decision Wheeler's
   framing asks about is not an event in the mathematics, so there is nothing for a later
   choice to reach back and alter.

2. **Nothing had become a record yet.** The detector qubit was correlated with the path,
   but that correlation was still a fully coherent, fully reversible piece of quantum
   state — sitting in the simulator's array, entangled with one other qubit and with
   nothing else in the world. Two gates undid it. A correlation that one operation can
   erase without a trace is not yet a *fact about the world*; it is a fact in waiting.

What makes an ordinary measurement irreversible is not that the wavefunction mystically
collapses at some particular instant. It is that the correlation spreads: from the
detector qubit into the amplifier, the cable, the screen, the photons bouncing off the
screen, the air, you. Undoing it would mean running *all* of that backwards in exact
reverse order. Nothing forbids it; it is just spectacularly, unbudgeably improbable — the
same kind of impossible as unstirring milk out of coffee.

The uncomfortable corollary is that the boundary between "reversible correlation" and
"irreversible record" is not a law of physics but a matter of scale and bookkeeping. Which
is exactly the subject of notebook 06, where the eraser is done properly: the record is
written into an *environment* of many qubits, and we watch how quickly it stops being
something anybody could undo.

## What you now know

- **A Hadamard is a 50/50 beam splitter**, `Rz` is a phase shifter, and `H · Rz(φ) · H` is
  a **Mach–Zehnder interferometer**. The circuits you were already writing had names in
  optics all along.
- $P(\text{port }0) = \cos^2(\varphi/2)$: both beam splitters are perfectly fair, yet one
  output port can be **completely dark**. Adding probabilities predicts a flat $1/2$ and
  is simply wrong. **Amplitudes add; probabilities do not**, and the minus sign in
  $H|1\rangle = (|0\rangle - |1\rangle)/\sqrt2$ is where the cancellation comes from.
- A **which-path detector** is a `CNOT`, and a *partial* one is a controlled rotation. As
  it learns more, the fringes fade — continuously, with no threshold, and **without anyone
  reading it**.
- **Complementarity is a conservation law**: $V^2 + D^2 = 1$, verified to machine
  precision. Fringe visibility and which-path knowledge are one resource split two ways,
  not two competing pictures of reality.
- The **Elitzur–Vaidman bomb tester** certifies a live bomb with a photon that was never
  absorbed: exploded $1/2$, certified $1/4$, inconclusive $1/4$, and a dud *never* fires
  port 1. What the bomb changed was not where the photon went but which histories stayed
  indistinguishable.
- **More paths sharpen the peak** without raising it. $2^k$ paths, phases in arithmetic
  progression, and the cancellation away from the peak gets sharper every time $k$ grows.
  That is the machinery of the QFT.
- **Stern–Gerlach filters in series**: `"zz"` passes everything, `"zx"` halves it, and
  `"zxz"` halves it *again* — a filter along an axis that had already been established.
  Measuring $x$ destroys $z$; the atoms do not carry definite values of both.
- **Delayed choice**: erasing the which-path record after the photon is already inside
  restores the fringes, and this is not retrocausality. The record had not yet become a
  record.

### What the qubit version does not capture

This notebook models interference honestly, but it models it with two paths and a qubit.
Three things that look like they should be here are genuinely out of reach, and it is
worth knowing which is which:

- **Hong–Ou–Mandel, and two-photon interference generally.** Send two identical photons
  into a beam splitter from opposite sides and they always leave together, by the same
  port — an interference effect between *photons*, not between paths. Describing it needs
  a mode that can hold 0, 1 or 2 photons (a bosonic Fock space), and a qubit holds one bit.
  No amount of qubits expresses it; it needs a different backend.
- **The literal double slit, with wave packets in space.** A real slit experiment has a
  continuous position variable, a spreading wave packet, and a fringe pattern smeared
  across a screen. What we ran captures the *logic* of that experiment — two
  indistinguishable routes, amplitudes adding, which-path knowledge destroying the pattern
  — but not its spatial physics. The two-path model is the skeleton, not the animal.
- **A genuine path integral.** Feynman's formulation sums over *all* paths, continuously
  many of them. Section 4 gestures at it by going from 2 to 8 paths and letting you watch
  the peak sharpen; the limit is a different computation and a different course.

None of these change any conclusion above. They are the parts of the picture a state
vector over qubits cannot draw.

## Next: `06-decoherence.ipynb`

Every experiment in this notebook turned on the same hinge: **is there, anywhere, a record
of which path the photon took?** When there is not, amplitudes cancel and ports go dark.
When there is, they do not — and it never mattered whether anyone read it.

So far every record we have written has been small and tidy: one detector qubit, one
scratch qubit, erasable with two gates. The next notebook removes that convenience. A real
qubit is coupled to an **environment** — stray photons, phonons, fields, $10^{23}$ degrees
of freedom you did not allocate and cannot address — and each of those couplings is
another which-path detector, written the same way, with `CNOT`s.

You will watch the visibility $V$ of this notebook decay as environment qubits are added
one at a time, and you will see why it never comes back: not because anything new happens
to the state, but because undoing the record now means undoing all of it, in reverse order,
simultaneously. You will meet **pointer states** — the special states that the environment
copies rather than destroys, which is why the world you see is made of positions and not
of superpositions of them — and you will run a real **quantum eraser**, where the fringes
come back only for the subset of runs sorted by what the environment recorded.

The conclusion is the one this project keeps arriving at from new directions: the classical
world is not something bolted onto the quantum one. It is what the quantum world looks like
from the inside, once the records have been written.